In [1]:
import os

In [2]:
%pwd

'c:\\Users\\dibim\\OneDrive\\Desktop\\Chicken_disease_classification\\Classification_project-chicken_disease\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\dibim\\OneDrive\\Desktop\\Chicken_disease_classification\\Classification_project-chicken_disease'

In [5]:
import tensorflow as tf

In [6]:
model = tf.keras.models.load_model("artifacts/training/trained_model.h5")

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    train_data: Path
    all_params: dict
    param_image_size: list
    param_batch_size: int

In [8]:
from Chicken_disease_classification.constants import *
from Chicken_disease_classification.utils.common import read_yaml, create_directories, save_json

In [9]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_evaluation_config(self) -> EvaluationConfig:
        evaluation_config = EvaluationConfig(
            path_of_model = "artifacts/training/trained_model.h5",
            train_data = "artifacts/data_ingestion/Chicken-fecal-images",
            all_params = self.params,
            param_image_size = self.params.IMAGE_SIZE,
            param_batch_size = self.params.BATCH_SIZE
        )

        return evaluation_config

In [10]:
from urllib.parse import urlparse

In [15]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.param_image_size[:-1],
            batch_size=self.config.param_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.train_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)

    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

In [16]:
try:
    config = ConfigurationManager()
    evaluation_config = config.get_evaluation_config()
    evaluation = Evaluation(config=evaluation_config)
    evaluation.evaluation()
    evaluation.save_score()
except Exception as e:
    raise e

Found 116 images belonging to 2 classes.
8/8 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.7759 - loss: 7.2567
